In [5]:
import os
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Activation,
    Dropout,
    Flatten,
    Dense,
)
from tensorflow.keras import backend as K

train_dir = "train"
model_weights_path = "model.h5"
img_width, img_height = 150, 150

if K.image_data_format() == "channels_first":
    input_shape = (3, img_width, img_height)
else:
    input_shape = (img_width, img_height, 3)

model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=input_shape))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3)))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(64))
model.add(Activation("relu"))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation("sigmoid"))

model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])
model.load_weights(model_weights_path)

X = []
y = []
filenames = []

for i in range(10700, 10800):
    cat_name = f"cat.{i}.jpg"
    dog_name = f"dog.{i}.jpg"

    cat_path = os.path.join(train_dir, cat_name)
    dog_path = os.path.join(train_dir, dog_name)

    img_cat = load_img(cat_path, target_size=(img_width, img_height))
    arr_cat = img_to_array(img_cat) / 255.0
    X.append(arr_cat)
    y.append(0)
    filenames.append(cat_name)

    img_dog = load_img(dog_path, target_size=(img_width, img_height))
    arr_dog = img_to_array(img_dog) / 255.0
    X.append(arr_dog)
    y.append(1)
    filenames.append(dog_name)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

pred = model.predict(X, verbose=0).ravel()
y_pred = (pred >= 0.5).astype(int)

acc = accuracy_score(y, y_pred)
f1 = f1_score(y, y_pred, average="macro")

cat_idx = filenames.index("cat.10727.jpg")
dog_idx = filenames.index("dog.10727.jpg")

p_cat_is_dog = float(pred[cat_idx])
p_dog_is_dog = float(pred[dog_idx])

p_cat_is_cat = 1.0 - p_cat_is_dog

print("accuracy =", round(acc, 3))
print("f1_macro =", round(f1, 3))
print("P(cat.10727 -> cats) =", round(p_cat_is_cat, 3))
print("P(dog.10727 -> dogs) =", round(p_dog_is_dog, 3))


c:\Users\Misha\PycharmProjects\itmo_moodle_cv\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


accuracy = 0.76
f1_macro = 0.755
P(cat.10727 -> cats) = 0.928
P(dog.10727 -> dogs) = 0.978
